<img src="logo.png" alt="Vegeta" width="240">

# Submarine propeller — flow in 3D with particles, weak points, vibration and noise spectra, and design updates

The 120 mm propeller of the AUV in `13_submarine`, on its own, in depth: how it performs at three
operating points, how the pressure on its blades falls with speed (and where that meets the vapour
pressure), what the flow looks like in 3D and in motion, where the blade is weakest, what the wake of the
four stern fins does to it — load harmonics, their frequencies and amplitudes, the vibration they drive and
the noise they radiate — and, at the end, what to change. The design is one dictionary (`DESIGN`, section 1):
edit it, or let section 10 propose and evaluate changes, then run the notebook again.

```
1  design (edit here) ─► operating points (Boreas, motor, battery, the vehicle's resistance)
2  performance; pressure vs speed along the blade and with depth ─► cavitation inception speeds
3  the wake: four fin wakes (model) or the hull's own wake (OpenFOAM, rans_ksst_external)
4  rotor CFD at the operating points (OpenFOAM, rotor_mrf): thrust, torque, efficiency, blade pressure vs speed
5  the flow in 3D: streamlines and tracer particles (slipstream model, and the CFD field when it exists)
6  movies (OpenCV): the particles through the rotor at each operating point, one after the other
7  blade FEA (Talos): stress at top speed, the weak points, modes in water
8  vibration: load harmonics (orders, frequencies, amplitudes), frequency diagram, blade response, fatigue at the weak points
9  noise: steady (Gutin) and wake (dipole) tones, broadband, levels with distance, cavitation
10 design updates: rules on the results ─► candidate designs ─► evaluated side by side ─► the recommendation
```

The OpenFOAM cells run when you run them (`VEGETA_SKIP_OPENFOAM=1` skips them; everything else runs without
OpenFOAM, the 3D views and movies then use the slipstream model). Every coefficient is an explicit input.

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv
from tqdm.auto import tqdm
from IPython.display import Video
from vegeta import dedalus, talos, aeromant, boreas, chronos
from vegeta.boreas import wake
from vegeta.aeromant import movie
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.dedalus.examples import Propeller as PropellerCAD

RUNS = Path("_runs/submarine_propeller"); shutil.rmtree(RUNS, ignore_errors=True); (RUNS / "cad").mkdir(parents=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
WATER = boreas.SEA_WATER                              # rho 1025 kg/m^3, c 1500 m/s
RHO_W, NU_W, G = WATER.density, 1.05e-6, 9.81
P_ATM, P_VAP = 101325.0, 2300.0                       # Pa: atmosphere, vapour pressure of sea water at ~20 C

# ---- hand-copied from 13_submarine (the vehicle and its drive): keep in sync by hand, on purpose ----------
L_M, D_M, S_WET, S_BODY = 1.2, 0.18, 0.694, 0.600     # vehicle length, diameter [m], wetted and bare-body surface [m^2]
M_VEHICLE = 25.7                                      # kg (13: mass budget)
W_MEAN, T_DED, HOTEL_W = 0.15, 0.10, 25.0             # mean wake fraction, thrust deduction, hotel load [W]
motor = boreas.Motor("thruster 100KV", kv_rpm_per_volt=100, resistance_ohm=0.6, no_load_current_a=0.3, max_current_a=5.0, mass_kg=0.35)
battery = boreas.Battery("7S Li-ion 20 Ah", cells=7, capacity_ah=20.0, usable_fraction=0.85, mass_kg=6.0)

def resistance(V):
    """Deeply submerged: ITTC friction x (1 + k) on the body, 1.5 x friction x 1.3 on the appendages (13, section 3)."""
    Re = V * L_M / NU_W
    Cf = 0.075 / (math.log10(Re) - 2) ** 2
    k = 1.5 * (D_M / L_M) ** 1.5 + 7 * (D_M / L_M) ** 3
    return 0.5 * RHO_W * V**2 * (S_BODY * Cf * (1 + k) + (S_WET - S_BODY) * Cf * 1.5 * 1.3)

SPEEDS = {"survey": 1.0, "cruise": 1.5}               # m/s; "top speed" is found per design (full throttle)
DEPTHS = {"near surface": 5.0, "survey depth": 50.0, "rated depth": 200.0}

## 1. The design — edit here

`DESIGN` is the propeller; `MATERIALS` are the blade materials with their fatigue strength **in sea water**
(corrosion fatigue, far below the in-air value — inputs to replace with data for the real alloy and finish);
`CRITERIA` are the pass marks section 10 checks. The same dictionary drives the blade-element model (Boreas)
and the CAD (Dedalus `Propeller`), so what is analysed is what is drawn.

Section properties follow from the geometry: `cd0` from thickness (Hoerner: `2 Cf (1 + 2 t/c + 60 (t/c)^4)`),
zero-lift angle from camber (thin aerofoil), and the minimum pressure coefficient
`Cp_min ≈ −(3.4 t/c + 0.6 cl)` (the thickness term of NACA 00xx sections plus a thin-aerofoil suction term) —
a thicker blade is stronger and cavitates sooner.

In [ ]:
DESIGN = dict(diameter_mm=120.0, pitch_mm=100.0, blades=3, chord_root_mm=18.0, chord_max_mm=30.0, chord_tip_mm=12.0,
              thickness=0.12, camber=0.05, material="6061-T6")
MATERIALS = {   # E [MPa], nu, density [t/mm^3], yield / ultimate / fatigue in sea water at 1e8 cycles [MPa]
    "6061-T6":            dict(E=69000.0,  nu=0.33, rho=2.70e-9, yield_=240.0, ultimate=310.0, fatigue_sw=45.0),
    "NAB (CuAl10Ni5Fe4)": dict(E=120000.0, nu=0.32, rho=7.60e-9, yield_=270.0, ultimate=650.0, fatigue_sw=120.0),
    "316L":               dict(E=193000.0, nu=0.30, rho=8.00e-9, yield_=220.0, ultimate=520.0, fatigue_sw=100.0),
}
CRITERIA = dict(sf_yield_min=3.0,            # static, at the weakest point, top speed
                sf_fatigue_min=2.0,          # Goodman, wake harmonics at top speed, at the weakest point
                mode_margin_min=0.20,        # blade modes (wet) vs every significant excitation at the operating points
                hull_margin_min=0.20,        # hull frequencies vs the shaft force orders at the operating points
                tone_db_max=75.0,            # loudest tone at cruise, dB re 1 uPa at 1 m
                cavitation_free_depth=5.0)   # no suction-side cavitation at top speed this deep or deeper
HUB = dict(hub_diameter=24.0, hub_height=16.0, bore=8.0)       # mm, fixed by the shaft and the stern cone
D_MAX_MM = 140.0                             # the largest propeller the stern allows (clearance to the fins)
N_FINS, FIN_DEPTH, FIN_WIDTH_DEG = 4, 0.25, 12.0               # fin wakes: peak wake fraction added, 1-sigma width (model inputs)
ADDED_MASS = 0.65                            # wet / dry frequency of a thin blade (assumed)
ZETA_BLADE = 0.02                            # damping ratio of the blade modes in water (assumed)
NOISE_ANGLE_DEG = 45.0                       # listening direction from the shaft axis
BLADE_ELEMENT_MM = 2.0

def section(d):
    cd0 = 2 * 0.0081 * (1 + 2 * d["thickness"] + 60 * d["thickness"] ** 4)
    return boreas.Airfoil(name=f"t/c {d['thickness']:.2f}, camber {d['camber']:.2f}", cl_alpha=5.5, alpha0_deg=-40.0 * d["camber"],
                          cl_max=1.0, cd0=cd0, k=0.05, source="thin aerofoil + Hoerner thickness drag")

def prop_of(d):
    return boreas.Propeller.from_pitch(f"{d['diameter_mm']:.0f} mm, {d['blades']} blades", d["diameter_mm"] / 1000, d["pitch_mm"] / 1000,
                                       blades=d["blades"], chord_root_m=d["chord_root_mm"] / 1000, chord_max_m=d["chord_max_mm"] / 1000,
                                       chord_tip_m=d["chord_tip_mm"] / 1000)

def cad_kw(d, blades=None):
    return dict(diameter=d["diameter_mm"], pitch=d["pitch_mm"], blades=blades or d["blades"], chord_root=d["chord_root_mm"],
                chord_max=d["chord_max_mm"], chord_tip=d["chord_tip_mm"], thickness=d["thickness"], camber=d["camber"], stations=8, **HUB)

def cp_min(d, cl):
    return -(3.4 * d["thickness"] + 0.6 * np.asarray(cl))

def material(d):
    m = MATERIALS[d["material"]]
    return talos.Material(d["material"], youngs_modulus=m["E"], poissons_ratio=m["nu"], density=m["rho"], yield_strength=m["yield_"], source="handbook")

def operating_points(d):
    """Propeller, section, drive and the three operating points (survey, cruise, top speed) of design d."""
    pr, sec = prop_of(d), section(d)
    drv = boreas.Propulsion(pr, sec, motor, battery, rho=RHO_W)
    pts = {k: drv.for_thrust(resistance(v) / (1 - T_DED), (1 - W_MEAN) * v) for k, v in SPEEDS.items()}
    lo, hi = 0.3, 6.0
    for _ in range(14):                                                   # top speed: full-throttle thrust = resistance
        m = 0.5 * (lo + hi)
        lo, hi = (m, hi) if drv.at_throttle(1.0, (1 - W_MEAN) * m).thrust * (1 - T_DED) > resistance(m) else (lo, m)
    pts["top speed"] = drv.at_throttle(1.0, (1 - W_MEAN) * lo)
    return pr, sec, drv, pts, dict(SPEEDS, **{"top speed": lo})

prop, sec, drive, pts, speeds = operating_points(DESIGN)
R_M = prop.radius
pd.DataFrame({k: {"boat_speed_m_s": speeds[k], "inflow_m_s": (1 - W_MEAN) * speeds[k], "rpm": v.rpm, "thrust_N": v.thrust, "torque_Nm": v.aero.torque,
                  "J": v.aero.advance_ratio, "prop_efficiency": v.aero.efficiency, "electrical_W": v.electrical_power, "current_A": v.current,
                  "current_limited": v.current_limited} for k, v in pts.items()}).round(3)

## 2. Performance, and pressure against speed

The open-water curves (thrust and torque coefficients, efficiency against advance ratio) at the cruise rpm,
then the pressure side of the story: along the blade the relative speed `V_rel = √(V_a² + (ωr)²)` sets the
dynamic pressure `q = ½ρV_rel²`, and the lowest pressure on the suction side is `p_static + q·Cp_min`.
When that falls to the vapour pressure the blade cavitates. Depth raises `p_static`; speed raises `q` with
its square — the second plot is the whole relation, one line per depth.

In [ ]:
Js, KT, KQ, ETA = [], [], [], []
n_c = pts["cruise"].rpm / 60
for v in np.linspace(0.05, 1.3, 30) * n_c * prop.diameter:
    op = boreas.solve(prop, sec, pts["cruise"].rpm, v, RHO_W)
    Js.append(op.advance_ratio); KT.append(op.thrust / (RHO_W * n_c**2 * prop.diameter**4))
    KQ.append(op.torque / (RHO_W * n_c**2 * prop.diameter**5)); ETA.append(op.efficiency)
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(Js, KT, label="K_T"); ax.plot(Js, 10 * np.array(KQ), label="10 K_Q"); ax.plot(Js, ETA, label="η₀")
for k, v in pts.items():
    ax.axvline(v.aero.advance_ratio, color="#888", ls=":"); ax.text(v.aero.advance_ratio, 0.02, f" {k}", rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="advance ratio J", ylim=(0, None), title=f"open-water curves at {pts['cruise'].rpm:.0f} rpm"); ax.legend(); ax.grid(alpha=0.3)

In [ ]:
def suction_peak(d, pt, inflow, depth):
    """Along the blade: r/R, relative speed, dynamic pressure and the suction-side minimum pressure (absolute, Pa)."""
    op = pt.aero
    omega = op.rpm * 2 * math.pi / 60
    v_rel = np.hypot(inflow + op.induced_velocity, omega * op.r)
    q = 0.5 * RHO_W * v_rel ** 2
    return op.r / (d["diameter_mm"] / 2000), v_rel, q, P_ATM + RHO_W * G * depth + q * cp_min(d, op.cl)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for k, v in pts.items():
    rR, vrel, q, pmin = suction_peak(DESIGN, v, (1 - W_MEAN) * speeds[k], DEPTHS["near surface"])
    ax[0].plot(rR, pmin / 1000, label=f"{k} ({v.rpm:.0f} rpm)")
ax[0].axhline(P_VAP / 1000, color="#c62828", ls="--", label="vapour pressure")
ax[0].set(xlabel="r / R", ylabel="lowest pressure on the blade [kPa abs]", title=f"suction peak along the blade at {DEPTHS['near surface']:.0f} m")
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
# pressure vs speed: the suction peak at 0.7 R over the whole speed range, one line per depth
sweep_V = np.linspace(0.5, speeds["top speed"], 14)
sweep = [drive.for_thrust(resistance(V) / (1 - T_DED), (1 - W_MEAN) * V) for V in sweep_V]
inception = {}
for name, depth in DEPTHS.items():
    p07 = np.array([np.interp(0.7, *suction_peak(DESIGN, s, (1 - W_MEAN) * V, depth)[::3]) for V, s in zip(sweep_V, sweep)])
    ax[1].plot(sweep_V, p07 / 1000, label=f"{name} ({depth:.0f} m)")
    below = np.where(p07 <= P_VAP)[0]
    inception[name] = float(sweep_V[below[0]]) if len(below) else None
ax[1].axhline(P_VAP / 1000, color="#c62828", ls="--", label="vapour pressure")
ax[1].set(xlabel="boat speed [m/s]", ylabel="lowest pressure at 0.7 R [kPa abs]", yscale="log", title="pressure vs speed: where cavitation starts")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3, which="both")
fig.tight_layout()
pd.DataFrame({name: {"depth_m": depth, "cavitation_inception_speed_m_s": inception[name] if inception[name] else f"none up to {speeds['top speed']:.2f}",
                     "sigma_at_top_speed_0.7R": boreas.cavitation(prop, pts["top speed"].rpm, (1 - W_MEAN) * speeds["top speed"], depth, WATER,
                                                                 cp_min=float(np.interp(0.7 * R_M, pts["top speed"].aero.r, cp_min(DESIGN, pts["top speed"].aero.cl))))["cavitation_number"]}
              for name, depth in DEPTHS.items()}).T

## 3. The wake the propeller works in

The propeller sits behind four cruciform fins; every blade crosses four fin wakes per revolution. The model
is a mean wake plus four Gaussian deficits (`N_FINS`, `FIN_DEPTH`, `FIN_WIDTH_DEG` — inputs). With OpenFOAM the
hull is run on its own (`rans_ksst_external`, the whole vehicle at cruise, nose upstream) and the wake is
read on the propeller plane just behind the stern cone; that measured wake then replaces the model in
sections 8 and 9.

In [ ]:
WAKE = wake.fin_wake(N_FINS, W_MEAN, FIN_DEPTH, FIN_WIDTH_DEG)
X_PROP = 0.015                                    # m: propeller plane behind the tail tip (vehicle frame of the CFD)
hull_cfd = None
if RUN_CFD:
    sub = dedalus.load_design("designs/submarine.py:Submarine")
    vehicle = sub.generate()
    nose_up = dedalus.Geometry.from_cadquery(vehicle.shape.rotate((0, 0, 0), (0, 0, 1), 180), name="vehicle_nose_upstream")
    stl_hull = nose_up.export_stl(RUNS / "cad" / "vehicle_nose_upstream.stl", tolerance=0.2)
    hull_case = aeromant.CFDCase("rans_ksst_external", stl_hull,
        dict(velocity=SPEEDS["cruise"], kinematic_viscosity=NU_W, density=RHO_W, reference_area=S_WET, reference_length=L_M / 4,
             center_of_rotation=(-0.64, 0.0, 0.0), iterations=400, residual_target=1e-4, cells_per_length=3.0, surface_level=4, near_level=3, wake_level=3),
        workdir=RUNS / "cfd_hull", geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    print(hull_case.prepare(overwrite=True))
    hull_cfd = hull_case.run(progress=True)
    print(hull_cfd)
    if hull_cfd.ok:
        WAKE = wake.sampled_wake(movie.openfoam_sampler(hull_case), (X_PROP, 0.0, 0.0), R_M, SPEEDS["cruise"],
                                 source="hull CFD (rans_ksst_external) at cruise, propeller plane")
else:
    print("hull CFD skipped (VEGETA_SKIP_OPENFOAM=1): the fin-wake model is used")
print("wake:", WAKE.source, f"| mean at 0.7 R {WAKE.mean():.3f}")
fig = plt.figure(figsize=(13, 4.2))
axp = fig.add_subplot(1, 2, 1, projection="polar")
rr, pp = np.meshgrid(np.linspace(0.3, 1.0, 15), np.radians(np.arange(0, 361, 3)))
pcm = axp.pcolormesh(pp, rr, WAKE(rr, np.degrees(pp)), cmap="viridis", shading="auto")
axp.set_title("axial wake fraction at the propeller plane"); fig.colorbar(pcm, ax=axp, fraction=0.04)
axh = fig.add_subplot(1, 2, 2)
hw = WAKE.harmonics(0.7, 16)
axh.bar(np.arange(1, 17), hw); axh.set(xlabel="circumferential order", ylabel="wake harmonic amplitude", title="wake harmonics at 0.7 R"); axh.grid(alpha=0.3)
fig.tight_layout()

## 4. CFD of the propeller at the operating points (OpenFOAM, rotating frame)

`rotor_mrf` in sea water at each operating point, with the mean inflow the propeller sees (`(1 − w)·V`).
Thrust, torque and efficiency against blade element theory, and — from the blade surfaces — the pressure
against the local relative speed: every face of the blade is a point; the lines are Bernoulli's `−q` and
`−½q`, and the dashed ones where the pressure would reach the vapour pressure at each depth.

In [ ]:
CFD_POINTS = ["survey", "cruise", "top speed"]        # trim this list to save time
CAD_KW = cad_kw(DESIGN)
prop_cad = PropellerCAD().generate(**CAD_KW)
pfiles = prop_cad.export(RUNS / "cad", formats=("step", "stl"), stl_tolerance=0.02)
prop_x = dedalus.Geometry.from_cadquery(prop_cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")   # axis z -> +x
stl_x = prop_x.export_stl(RUNS / "cad" / "prop_axis_x.stl", tolerance=0.02)
prop_mesh_m = dviz.to_pyvista(prop_x, 0.05).scale(0.001, inplace=False)            # for the 3D scenes, in metres
dviz.show(dviz.plot3d(prop_cad))
cfd_cases, cfd_res = {}, {}
if RUN_CFD:
    for name in CFD_POINTS:
        v = pts[name]
        params = dict(rpm=v.rpm, airspeed=(1 - W_MEAN) * speeds[name], diameter=prop.diameter, kinematic_viscosity=NU_W, density=RHO_W, rotation=1,
                      iterations=400, cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)
        cfd_cases[name] = aeromant.CFDCase("rotor_mrf", stl_x, params, workdir=RUNS / f"cfd_{name.replace(' ', '_')}", geometry_units="mm",
                                           environment=aeromant.OpenFOAMEnvironment.detect())
        cfd_cases[name].prepare(overwrite=True)
        cfd_res[name] = cfd_cases[name].run(progress=True)
        print(name, cfd_res[name].status, {k: round(cfd_res[name].metrics.get(k, float("nan")), 4) for k in ("thrust_N", "torque_Nm", "efficiency")})
else:
    print("propeller CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM")
CFD_OK = {k: r for k, r in cfd_res.items() if r.ok}

In [ ]:
if CFD_OK:
    rows = {}
    for name, r in CFD_OK.items():
        v = pts[name]
        rows[name] = {"thrust BEMT": v.thrust, "thrust CFD": r.metrics["thrust_N"], "torque BEMT": v.aero.torque, "torque CFD": r.metrics["torque_Nm"],
                      "efficiency BEMT": v.aero.efficiency, "efficiency CFD": r.metrics["efficiency"], "converged": r.metrics["converged"]}
    display(pd.DataFrame(rows).T.round(4))
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    vs = [speeds[k] for k in CFD_OK]
    for i, (key, lab) in enumerate((("thrust", "thrust [N]"), ("torque", "torque [N m]"), ("efficiency", "efficiency"))):
        ax[i].plot(sweep_V, [getattr(s, "thrust") if key == "thrust" else (s.aero.torque if key == "torque" else s.aero.efficiency) for s in sweep], label="BEMT")
        ax[i].plot(vs, [rows[k][f"{key} CFD"] for k in CFD_OK], "o", label="CFD")
        ax[i].set(xlabel="boat speed [m/s]", ylabel=lab); ax[i].legend(); ax[i].grid(alpha=0.3)
    fig.tight_layout()

def blade_surface_pressure(case, rotor):
    """Faces of the rotor surface: local relative speed [m/s] and static pressure [Pa, gauge]."""
    b = aviz.read_results(case)["boundary"]
    V_rel, p = [], []
    for i in range(b.n_blocks):
        name = b.get_block_name(i)
        if name and name.startswith("body"):
            blk = b[i]
            c = np.asarray(blk.cell_centers().points) - np.asarray(rotor.center)
            r = np.hypot(c[:, 1], c[:, 2])
            V_rel.append(np.hypot(rotor.inflow, abs(rotor.omega) * r)); p.append(np.asarray(blk.cell_data["p"]) * RHO_W)   # kinematic -> Pa
    return np.concatenate(V_rel), np.concatenate(p)

if CFD_OK:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    vmax = 0.0
    for name in CFD_OK:
        rv = movie.RotorView.from_case(cfd_cases[name], DESIGN["blades"])
        Vr, p = blade_surface_pressure(cfd_cases[name], rv)
        ax.scatter(Vr, p / 1000, s=3, alpha=0.4, label=name); vmax = max(vmax, Vr.max())
    vv = np.linspace(0, vmax, 50)
    ax.plot(vv, -0.5 * RHO_W * vv**2 / 1000, "k-", lw=1, label="−q"); ax.plot(vv, -0.25 * RHO_W * vv**2 / 1000, "k--", lw=1, label="−q/2")
    for dname, depth in DEPTHS.items():
        ax.axhline((P_VAP - P_ATM - RHO_W * G * depth) / 1000, ls=":", color="#c62828")
        ax.text(0, (P_VAP - P_ATM - RHO_W * G * depth) / 1000, f" vapour at {depth:.0f} m", color="#c62828", fontsize=8, va="bottom")
    ax.set(xlabel="local relative speed [m/s]", ylabel="blade surface pressure [kPa gauge]", title="pressure vs speed on the blades (CFD)")
    ax.legend(fontsize=8, markerscale=4); ax.grid(alpha=0.3)
    aviz.show(aviz.plot_surface_pressure(cfd_cases[list(CFD_OK)[-1]]))

## 5. The flow in 3D, with particles

Streamlines through the disc and a cloud of tracer particles (each with a short trail), coloured by speed,
around the propeller. The first scene uses the slipstream model of the blade-element solution
(`boreas.wake.slipstream_sampler`: induced axial velocity and swirl, contraction by continuity); when the
CFD has run, the same scene is drawn from the OpenFOAM field. Rotate and zoom in a live kernel.

In [ ]:
def field_grid(sampler, center, D, n=(60, 28, 28), x_range=(-1.2, 3.0), lateral=0.8):
    """The field on a regular grid (for streamlines): U and |U|, zero where the sampler has no flow."""
    c = np.asarray(center, float)
    lo = c + np.array([x_range[0] * D, -lateral * D, -lateral * D])
    sp = np.array([(x_range[1] - x_range[0]) * D / (n[0] - 1), 2 * lateral * D / (n[1] - 1), 2 * lateral * D / (n[2] - 1)])
    grid = pv.ImageData(dimensions=n, spacing=tuple(sp), origin=tuple(lo))
    U, ok = sampler(np.asarray(grid.points))
    U = np.where(ok[:, None], U, 0.0)
    grid.point_data["U"] = U
    grid.point_data["|U|"] = np.linalg.norm(U, axis=1)
    return grid

def particle_scene(sampler, rotor, title, n=160, steps=70, degrees_per_frame=8):
    """Streamlines + tracer particles with trails + the propeller, in one pyvista scene."""
    D, c = rotor.diameter, np.asarray(rotor.center, float)
    tr = movie.Tracer(sampler, rotor, n=n, seed=3, trail=14)
    dt = movie.time_step(rotor, degrees_per_frame)
    for _ in range(steps):
        tr.step(dt)
    grid = field_grid(sampler, c, D)
    seeds = pv.Disc(center=tuple(c - np.array([0.9 * D, 0, 0])), inner=0.08 * D, outer=0.55 * D, normal=(1, 0, 0), r_res=4, c_res=16)
    stream = grid.streamlines_from_source(seeds, vectors="U", max_steps=4000)
    vmax = float(np.percentile(grid.point_data["|U|"][grid.point_data["|U|"] > 0], 99))
    pl = pv.Plotter(window_size=(1100, 650))
    if stream.n_points:
        pl.add_mesh(stream.tube(radius=0.004 * D), scalars="|U|", cmap="turbo", clim=(0, vmax), opacity=0.35, show_scalar_bar=False)
    L = tr.trails.shape[0]
    pts_ = tr.trails.transpose(1, 0, 2).reshape(-1, 3)
    lines = np.hstack([[L] + list(range(j * L, (j + 1) * L)) for j in range(tr.n)])
    trails = pv.PolyData(pts_, lines=lines)
    trails.point_data["|U|"] = np.repeat(tr.speed, L)
    pl.add_mesh(trails, scalars="|U|", cmap="turbo", clim=(0, vmax), line_width=2, show_scalar_bar=False)
    cloud = pv.PolyData(tr.pos); cloud.point_data["|U|"] = tr.speed
    pl.add_mesh(cloud, scalars="|U|", cmap="turbo", clim=(0, vmax), render_points_as_spheres=True, point_size=11, scalar_bar_args={"title": "|U| [m/s]"})
    pl.add_mesh(prop_mesh_m.translate(tuple(c), inplace=False), color="#9a9a9a", smooth_shading=True)
    pl.add_text(title, font_size=10); pl.add_axes()
    pl.camera_position = [tuple(c + D * np.array([-1.6, -2.6, 1.4])), tuple(c + D * np.array([0.8, 0, 0])), (0, 0, 1)]
    return pl

def model_field(pt, V_in):
    """The slipstream model of an operating point and the rotor view that goes with it (centre at the origin)."""
    aero = boreas.solve(prop, sec, pt.rpm, V_in, RHO_W)
    rot = movie.RotorView(center=(0.0, 0.0, 0.0), diameter=prop.diameter, rpm=pt.rpm, rotation=1, blades=DESIGN["blades"], inflow=V_in)
    return wake.slipstream_sampler(prop, aero, center=rot.center), rot

sampler_m, rotor_m = model_field(pts["cruise"], (1 - W_MEAN) * speeds["cruise"])
aviz.show(particle_scene(sampler_m, rotor_m, f"cruise, {pts['cruise'].rpm:.0f} rpm: slipstream model (streamlines, tracer particles)"))

In [ ]:
if CFD_OK:
    name = "cruise" if "cruise" in CFD_OK else list(CFD_OK)[0]
    s_cfd = movie.openfoam_sampler(cfd_cases[name])
    r_cfd = movie.RotorView.from_case(cfd_cases[name], DESIGN["blades"])
    aviz.show(particle_scene(s_cfd, r_cfd, f"{name}: OpenFOAM field (rotor_mrf), tracer particles"))
    aviz.show(aviz.plot_streamlines(cfd_cases[name], n=80, normal_plane="z"))

## 6. Movies: how the flow changes with the operating point

A few tracer particles through the rotor, side view and along the axis, the blades turning in consistent
slow motion (`vegeta.aeromant.movie`, OpenCV). One clip per operating point, joined into one movie —
survey, cruise, top speed: the slipstream accelerates and the swirl tightens as the rpm rises. With OpenFOAM
the same is made from the CFD fields.

In [ ]:
clips = []
for name in ("survey", "cruise", "top speed"):
    s_, r_ = model_field(pts[name], (1 - W_MEAN) * speeds[name])
    clips.append(movie.make_movie(None, RUNS / f"model_{name.replace(' ', '_')}.mp4", rotor=r_, sampler=s_, n=40, seconds=4, fps=24,
                                  degrees_per_frame=10, title=f"slipstream model, {name}: {speeds[name]:.2f} m/s, {pts[name].rpm:.0f} rpm"))
flow_model = movie.concat_videos(clips, RUNS / "flow_by_operating_point_model.mp4", captions=["survey", "cruise", "top speed"])
display(Video(str(flow_model), embed=False, width=960))

In [ ]:
if CFD_OK:
    cfd_clips = [movie.make_movie(cfd_cases[k], RUNS / f"cfd_{k.replace(' ', '_')}.mp4", blades=DESIGN["blades"], n=40, seconds=5, fps=24, degrees_per_frame=10,
                                  title=f"OpenFOAM rotor_mrf, {k}: {speeds[k]:.2f} m/s, {pts[k].rpm:.0f} rpm") for k in CFD_OK]
    flow_cfd = movie.concat_videos(cfd_clips, RUNS / "flow_by_operating_point_cfd.mp4", captions=list(CFD_OK))
    display(Video(str(flow_cfd), embed=False, width=960))

## 7. Blade strength and the weak points (Talos)

One blade with its hub, held at the shaft bore (so the hub carries the load into the root as the real part
does), the blade's share of the top-speed thrust and torque applied over the blade surface. The weak points
are the separate stress hotspots (the highest nodes at least 6 mm apart), each located by radius, chordwise
position (0 = trailing edge, 1 = leading edge, for rotation +1) and face. Note: the CAD has no root fillet;
a hotspot at the blade–hub junction is that sharp corner, and the first design change it calls for is a fillet.

In [ ]:
def blade_fea(d, top, tag):
    """Static (top speed) and modal blade analysis of design d; returns (blade CAD, model, static result, modes result, loads)."""
    kw = cad_kw(d, blades=1)
    blade = PropellerCAD().generate(**kw)
    f = blade.export(RUNS / f"cad_blade_{tag}", formats=("step",))
    rb, hh, R = kw["bore"] / 2 + 0.5, kw["hub_height"], kw["diameter"] / 2
    regions = [talos.SurfacesInBox("bore", (-rb, -rb, -hh / 2 - 0.5, rb, rb, hh / 2 + 0.5)),
               talos.SurfacesInBox("blade", (kw["hub_diameter"] / 2 - 2.0, -R, -R, R + 1.0, R, R))]
    Tb = top.thrust / d["blades"]
    Ft = top.aero.torque / (d["blades"] * 0.7 * R / 1000)
    m = talos.StructuralModel(f.artifacts["step"], "mm-N-MPa", material(d), regions, [talos.FixedSupport("bore")],
                              [talos.Force("blade", fz=Tb, fy=-Ft)], talos.MeshSettings(element_size=BLADE_ELEMENT_MM), name=f"blade_{tag}")
    m.mesh(RUNS / f"blade_fea_{tag}")
    res = m.solve(RUNS / f"blade_fea_{tag}")
    shutil.copytree(RUNS / f"blade_fea_{tag}", RUNS / f"blade_modal_{tag}", dirs_exist_ok=True)
    modes = m.solve_modes(RUNS / f"blade_modal_{tag}", n_modes=4)
    return blade, m, res, modes, {"thrust_per_blade_N": Tb, "tangential_per_blade_N": Ft}

def propeller_mass_g(d, one_blade_with_hub_mm3):
    """Mass of the whole propeller from the one-blade CAD (blade + hub): the hub counted once."""
    hub = math.pi * ((HUB["hub_diameter"] / 2) ** 2 - (HUB["bore"] / 2) ** 2) * HUB["hub_height"]
    return ((one_blade_with_hub_mm3 - hub) * d["blades"] + hub) * MATERIALS[d["material"]]["rho"] * 1e6

def weak_points(res, d, n=6, frac=0.25, spacing=6.0):
    """Separate stress hotspots: location, stress, static safety factor."""
    fr = talos.read_frd(res.artifacts["frd"])
    x, y, z = fr.coords.T
    r, vm = np.hypot(x, y), fr.von_mises
    hub_r, R = HUB["hub_diameter"] / 2, d["diameter_mm"] / 2
    cand = np.where(vm >= frac * vm.max())[0]
    cand = cand[np.argsort(-vm[cand])]
    picked = []
    for i in cand:
        if all(np.linalg.norm(fr.coords[i] - fr.coords[j]) > spacing for j in picked):
            picked.append(i)
        if len(picked) >= n:
            break
    rows = []
    blade_nodes = r > hub_r + 0.3
    for k, i in enumerate(picked):
        sl = blade_nodes & (np.abs(r - r[i]) < 1.0)
        chord = float((y[i] - y[sl].min()) / max(np.ptp(y[sl]), 1e-9)) if sl.sum() > 3 else float("nan")
        face = "forward (suction) face" if sl.sum() > 3 and z[i] >= z[sl].mean() else "aft (pressure) face"
        if r[i] <= hub_r + 0.6:
            where = "blade-hub junction (root corner)"
        elif r[i] / R > 0.85:
            where = "tip region"
        elif chord < 0.12:
            where = "trailing edge"
        elif chord > 0.88:
            where = "leading edge"
        else:
            where = "blade root area" if r[i] / R < 0.35 else "mid-chord"
        rows.append({"id": f"W{k + 1}", "where": where, "r_over_R": r[i] / R, "chord_position": chord, "face": face,
                     "von_mises_MPa": float(vm[i]), "SF_yield": MATERIALS[d["material"]]["yield_"] / float(vm[i]), "xyz_mm": fr.coords[i].round(2).tolist()})
    return pd.DataFrame(rows).set_index("id")

blade, blade_model, res_blade, blade_modes, blade_loads = blade_fea(DESIGN, pts["top speed"], "base")
print(res_blade)
weak = weak_points(res_blade, DESIGN)
f_air = blade_modes.metrics["frequencies_hz"]
f_wet = [f * ADDED_MASS for f in f_air]
print(f"blade loads at top speed ({pts['top speed'].rpm:.0f} rpm): {blade_loads['thrust_per_blade_N']:.2f} N thrust, "
      f"{blade_loads['tangential_per_blade_N']:.2f} N tangential per blade | propeller mass {propeller_mass_g(DESIGN, blade.volume):.0f} g")
print("blade modes in air:", [round(f) for f in f_air], "Hz; in water:", [round(f) for f in f_wet], "Hz")
weak.round(3)

In [ ]:
pl = tviz.plot_results(res_blade, field="von_mises")
hot = np.array([p for p in weak["xyz_mm"]])
span = float(np.ptp(talos.read_frd(res_blade.artifacts["frd"]).coords, axis=0).max())
pl.add_mesh(pv.PolyData(hot), color="#d50000", render_points_as_spheres=True, point_size=16)
pl.add_point_labels(hot + np.array([0.0, 0.0, 0.04 * span]), list(weak.index), font_size=14, point_size=1, shape_opacity=0.6)
aviz.show(pl)
tviz.show(tviz.plot_mode(blade_modes, mode=1))
video = tviz.animate(res_blade, RUNS / "blade_stress.mp4", rpm=pts["top speed"].rpm, axis="z", seconds=5, fps=24)   # load ramps 0 -> top speed while it turns
display(Video(str(video), embed=False, width=720))

## 8. Vibration: frequencies and amplitudes

Behind four fins every blade load pulses four times per revolution; what reaches the shaft depends on the
blade count (`boreas.wake.load_harmonics`): the thrust only at common multiples of blades and fins, the
side (bearing) forces one order either side. The tables and bars give every order's frequency and
amplitude; the frequency diagram puts them against the blade modes (in water) and the hull's own
frequencies; the blade response multiplies each blade-load harmonic by the static response per newton
(FEA) and the dynamic amplification of the nearest mode.

In [ ]:
H = {k: wake.load_harmonics(prop, sec, v.rpm, speeds[k], WAKE, RHO_W) for k, v in pts.items()}
significant = {}
for k, h in H.items():
    t = pd.DataFrame(h.table(30)).set_index("order")
    significant[k] = t[(t.drop(columns="frequency_hz") > 1e-3 * h.amplitudes["shaft_thrust_N"][0]).any(axis=1)]
print("orders with a load above 0.1 % of the mean thrust, top speed:")
display(significant["top speed"].round(4))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i, (col, lab) in enumerate((("blade_thrust_N", "blade thrust"), ("shaft_thrust_N", "shaft thrust"), ("side_force_y_N", "side force (y)"))):
    for j, (k, h) in enumerate(H.items()):
        t = pd.DataFrame(h.table(30))
        ax[i].bar(t["frequency_hz"] + (j - 1) * 3, t[col], width=3, label=k)
    ax[i].set(xlabel="frequency [Hz]", ylabel="amplitude [N]", title=f"{lab} harmonics", yscale="log", ylim=(1e-4, None)); ax[i].grid(alpha=0.3); ax[i].legend(fontsize=8)
fig.tight_layout()

In [ ]:
# the hull's own frequencies: first free-free bending of the vehicle as a uniform beam (pressure hull + fairing), and the pressure-hull ring mode (13)
EI = 69e9 * math.pi * 0.075**3 * 0.006 + 20e9 * math.pi * 0.09**3 * 0.002      # N m^2: aluminium tube r 75 t 6 mm + GFRP fairing r 90 t 2 mm
f_girder = 4.730**2 / (2 * math.pi * L_M**2) * math.sqrt(EI / (M_VEHICLE / L_M))
f_ring = 11400.0                                                                  # Hz, 13_submarine section 6
blade_structure = chronos.Structure(tuple(f_wet), damping_ratio=ZETA_BLADE, source="Talos modal x added-mass factor")
hull_structure = chronos.Structure((f_girder, f_ring), damping_ratio=0.03, source="beam estimate + ring mode")
B = DESIGN["blades"]
orders = {"blade load (fin wakes)": sorted({int(o) for o in significant["top speed"].index if significant["top speed"].loc[o, "blade_thrust_N"] > 0}),
          "shaft thrust": sorted({int(o) for o in significant["top speed"].index if significant["top speed"].loc[o, "shaft_thrust_N"] > 0}),
          "side force": sorted({int(o) for o in significant["top speed"].index if significant["top speed"].loc[o, "side_force_y_N"] > 0}),
          "blade pass (steady)": [B, 2 * B]}
styles = {"blade load (fin wakes)": ("#2e7d32", "-"), "shaft thrust": ("#1565c0", "-"), "side force": ("#6a1b9a", "--"), "blade pass (steady)": ("#555555", ":")}
rpm_axis = np.linspace(100, 1.15 * pts["top speed"].rpm, 60)
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(rpm_axis, rpm_axis / 60, color="#999", lw=1, label="1P")
for kind, ords in orders.items():
    col, ls = styles[kind]
    for j, o in enumerate(ords):
        ax.plot(rpm_axis, o * rpm_axis / 60, color=col, ls=ls, lw=1.2, label=kind if j == 0 else None)
        ax.text(rpm_axis[-1], o * rpm_axis[-1] / 60, f" {o}P", color=col, fontsize=7, va="center")
for i, f in enumerate(f_wet):
    ax.axhline(f, color="#c62828", lw=1); ax.text(rpm_axis[0], f, f" blade mode {i + 1} (wet) {f:.0f} Hz", color="#c62828", fontsize=8, va="bottom")
ax.axhline(f_girder, color="#ef6c00", lw=1); ax.text(rpm_axis[0], f_girder, f" hull bending {f_girder:.0f} Hz", color="#ef6c00", fontsize=8, va="bottom")
for k, v in pts.items():
    ax.axvline(v.rpm, color="#888", ls=":"); ax.text(v.rpm, 2.0, f" {k}", rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="rpm", ylabel="frequency [Hz]", yscale="log", ylim=(1.5, 2 * max(f_wet)), title="frequency diagram: excitation orders vs blade and hull frequencies")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3, which="both")
margin_rows = []
for k, h in H.items():
    for kind, ords in orders.items():
        for o in ords:
            f = o * h.shaft_hz
            st = blade_structure if kind in ("blade load (fin wakes)", "blade pass (steady)") else hull_structure
            margin_rows.append({"point": k, "excitation": f"{kind} {o}P", "frequency_hz": f, "nearest_mode_hz": st.nearest_mode(f),
                                "margin": st.margin(f), "amplification": float(st.amplification(f)[0])})
margins = pd.DataFrame(margin_rows)
print(f"hull bending ~{f_girder:.0f} Hz (a uniform-beam estimate), ring mode {f_ring / 1000:.1f} kHz")
margins.sort_values("margin").head(8).round(3)

In [ ]:
# blade response: static response per newton of blade thrust (FEA, top-speed load case) x harmonic x amplification of the nearest wet mode
T_top = blade_loads["thrust_per_blade_N"]
u_per_N = res_blade.metrics["max_displacement"] / T_top                         # mm per N, tip
resp_rows = []
for k, h in H.items():
    for o in range(1, 31):
        A = h.amplitudes["blade_thrust_N"][o]
        if A <= 0:
            continue
        f = o * h.shaft_hz
        daf = float(blade_structure.amplification(f)[0])
        resp_rows.append({"point": k, "order": o, "frequency_hz": f, "blade_thrust_amp_N": A, "amplification": daf,
                          "tip_amplitude_um": 1000 * u_per_N * A * daf,
                          "stress_amplitude_MPa (weakest point)": weak["von_mises_MPa"].iloc[0] / T_top * A * daf})
response = pd.DataFrame(resp_rows)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k in H:
    rr_ = response[response.point == k]
    ax[0].stem(rr_["frequency_hz"], rr_["tip_amplitude_um"], linefmt="-", markerfmt="o", basefmt=" ", label=k)
    ax[1].stem(rr_["frequency_hz"], rr_["stress_amplitude_MPa (weakest point)"], linefmt="-", markerfmt="o", basefmt=" ", label=k)
ax[0].set(xlabel="frequency [Hz]", ylabel="tip amplitude [µm]", title="blade tip vibration"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="frequency [Hz]", ylabel="stress amplitude [MPa]", title=f"alternating stress at {weak.index[0]} ({weak['where'].iloc[0]})"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()
# fatigue at every weak point, every operating point: Goodman with the mean stress of the point and the summed harmonics (conservative)
mat = MATERIALS[DESIGN["material"]]
fat_rows = []
for k, v in pts.items():
    alt = response[response.point == k]
    for wid, w in weak.iterrows():
        s_per_N = w["von_mises_MPa"] / T_top
        s_m = s_per_N * v.thrust / DESIGN["blades"]
        s_a = s_per_N * float((alt["blade_thrust_amp_N"] * alt["amplification"]).sum())
        fat_rows.append({"point": k, "weak point": wid, "where": w["where"], "mean_MPa": s_m, "alternating_MPa": s_a,
                         "SF_fatigue (Goodman, sea water)": 1.0 / (s_a / mat["fatigue_sw"] + s_m / mat["ultimate"]) if s_a + s_m > 0 else float("inf")})
fatigue = pd.DataFrame(fat_rows)
fatigue.pivot(index="weak point", columns="point", values="SF_fatigue (Goodman, sea water)").round(2)

## 9. Noise: tones, broadband, distance, cavitation

Three sources, in dB re 1 µPa at 1 m, `NOISE_ANGLE_DEG` from the shaft axis: the steady-loading tones at
the blade-passing frequency and its harmonics (Gutin), the wake tones — the shaft force harmonics of
section 8 as dipoles, usually the loudest part of a propeller behind appendages — and a broadband allowance.
Levels at distance follow spherical spreading (20 log r; absorption is negligible below a few kHz). Above
cavitation inception none of this holds: collapsing bubbles dominate.

In [ ]:
def noise_spectrum(pr, pt, h, distance=1.0, angle=NOISE_ANGLE_DEG):
    steady = boreas.gutin_harmonics(pr, pt.thrust, pt.aero.torque, pt.rpm, distance, angle, WATER, harmonics=5)
    unsteady = wake.unsteady_tones(h, distance, WATER, angle_deg=angle)
    bb = boreas.broadband_level(pr, pt.thrust, pt.rpm, distance, WATER)
    tones = [(f, L, "steady (Gutin)") for f, L in zip(steady["frequency_hz"], steady["spl_db"])] + [(t["frequency_hz"], t["spl_db"], "wake (dipole)") for t in unsteady]
    total_tonal = 10 * math.log10(sum(10 ** (L / 10) for _, L, _ in tones)) if tones else -np.inf
    return tones, bb, total_tonal, 10 * math.log10(10 ** (total_tonal / 10) + 10 ** (bb / 10))

fig, axes = plt.subplots(1, 3, figsize=(17, 4), sharey=True)
noise_rows = {}
for ax, (k, v) in zip(axes, pts.items()):
    tones, bb, tonal, total = noise_spectrum(prop, v, H[k])
    for kind, col in (("steady (Gutin)", "#1565c0"), ("wake (dipole)", "#c62828")):
        f = [t[0] for t in tones if t[2] == kind]; L = [t[1] for t in tones if t[2] == kind]
        if f:
            ax.stem(f, L, linefmt=col, markerfmt="o", basefmt=" ", label=kind)
    ax.axhline(bb, color="#555", ls="--", label="broadband allowance")
    ax.set(xlabel="frequency [Hz]", title=f"{k}: {v.rpm:.0f} rpm, total {total:.0f} dB", ylim=(0, None)); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    loudest = max(tones, key=lambda t: t[1]) if tones else (float("nan"), -np.inf, "")
    cav = {name: boreas.cavitation(prop, v.rpm, (1 - W_MEAN) * speeds[k], depth, WATER, cp_min=float(np.interp(0.7 * R_M, v.aero.r, cp_min(DESIGN, v.aero.cl))))["cavitates"]
           for name, depth in DEPTHS.items()}
    noise_rows[k] = {"rpm": v.rpm, "loudest_tone_hz": loudest[0], "loudest_tone_db": loudest[1], "loudest_tone_source": loudest[2], "tonal_db": tonal,
                     "broadband_db": bb, "total_db_1m": total, "total_db_100m": total - 40.0, "total_db_1km": total - 60.0,
                     **{f"cavitates at {n}": c for n, c in cav.items()}}
axes[0].set_ylabel("dB re 1 µPa at 1 m")
fig.tight_layout()
noise = pd.DataFrame(noise_rows).T
noise

## 10. Design updates: what to change, and what it buys

`evaluate(design)` runs everything above that decides the design — operating points, wake harmonics,
blade FEA and modes, weak points, fatigue, tones, cavitation — in about twenty seconds, without CFD.
`propose(design, result)` reads the result against `CRITERIA` and returns candidate changes with their
reasons: a stronger root when a weak point fails, a blade count whose shaft orders miss the strong fin-wake
harmonics when the tones are loud or a shaft order meets a hull frequency, more pitch or diameter when the
blades cavitate, a thinner blade when there is strength to spare. A candidate that fails a criterion gets a
second round: the fixes the rules propose for it are applied and it is evaluated again. Every candidate is evaluated the same way and put side by side with the
baseline. Add your own ideas to `MY_CANDIDATES`; adopt one by editing `DESIGN` in section 1 and re-running
the notebook (then run its CFD, sections 3–6).

In [ ]:
def evaluate(d, tag):
    """The decision metrics of design d (no CFD)."""
    try:
        pr, sc, drv, p_, sp_ = operating_points(d)
    except ValueError as e:
        return {"feasible": False, "note": str(e)}
    hh = {k: wake.load_harmonics(pr, sc, v.rpm, sp_[k], WAKE, RHO_W) for k, v in p_.items()}
    bl, _, rs, md, loads = blade_fea(d, p_["top speed"], tag)
    wk = weak_points(rs, d, n=3)
    fw = [f * ADDED_MASS for f in md.metrics["frequencies_hz"]]
    st = chronos.Structure(tuple(fw), damping_ratio=ZETA_BLADE)
    exc = [(o * h.shaft_hz) for h in hh.values() for o in range(1, 31) if h.amplitudes["blade_thrust_N"][o] > 1e-3 * h.amplitudes["shaft_thrust_N"][0]]
    exc += [d["blades"] * v.rpm / 60 for v in p_.values()]
    shaft_exc = [o * h.shaft_hz for h in hh.values() for o in range(1, 31)
                 if max(h.amplitudes["shaft_thrust_N"][o], h.amplitudes["side_force_y_N"][o]) > 1e-3 * h.amplitudes["shaft_thrust_N"][0]]
    mat = MATERIALS[d["material"]]
    T_top = loads["thrust_per_blade_N"]
    h_top = hh["top speed"]
    s_per_N = wk["von_mises_MPa"].iloc[0] / T_top
    s_a = s_per_N * sum(h_top.amplitudes["blade_thrust_N"][o] * float(st.amplification(o * h_top.shaft_hz)[0]) for o in range(1, 31))
    s_m = s_per_N * p_["top speed"].thrust / d["blades"]
    tones, bb, tonal, total = noise_spectrum(pr, p_["cruise"], hh["cruise"])
    top = p_["top speed"]
    cpm = float(np.interp(0.7 * pr.radius, top.aero.r, cp_min(d, top.aero.cl)))
    cav_depth = next((dep for dep in np.arange(0.0, 300.0, 1.0)
                      if not boreas.cavitation(pr, top.rpm, (1 - W_MEAN) * sp_["top speed"], dep, WATER, cp_min=cpm)["cavitates"]), float("inf"))
    cr = p_["cruise"]
    return {"feasible": True, "blades": d["blades"], "diameter_mm": d["diameter_mm"], "pitch_mm": d["pitch_mm"], "thickness": d["thickness"],
            "chord_root_mm": d["chord_root_mm"], "material": d["material"],
            "cruise_rpm": cr.rpm, "cruise_efficiency": cr.aero.efficiency, "cruise_electrical_W": cr.electrical_power,
            "endurance_h": battery.usable_wh / (cr.electrical_power + HOTEL_W), "top_speed_m_s": sp_["top speed"],
            "blade_mass_g": propeller_mass_g(d, bl.volume),
            "weakest_point": wk["where"].iloc[0], "peak_stress_MPa": wk["von_mises_MPa"].iloc[0], "SF_yield": wk["SF_yield"].iloc[0],
            "SF_fatigue": 1.0 / (s_a / mat["fatigue_sw"] + s_m / mat["ultimate"]),
            "first_wet_mode_hz": fw[0], "mode_margin": min(st.margin(f) for f in exc),
            "hull_margin": min((hull_structure.margin(f) for f in shaft_exc), default=float("inf")),
            "loudest_tone_db": max(t[1] for t in tones), "loudest_tone_hz": max(tones, key=lambda t: t[1])[0], "total_db_cruise": total,
            "cavitation_free_below_m": cav_depth}

def passes(m):
    c = CRITERIA
    return {"strength": m["SF_yield"] >= c["sf_yield_min"], "fatigue": m["SF_fatigue"] >= c["sf_fatigue_min"],
            "modes": m["mode_margin"] >= c["mode_margin_min"], "hull modes": m["hull_margin"] >= c["hull_margin_min"], "noise": m["loudest_tone_db"] <= c["tone_db_max"],
            "cavitation": m["cavitation_free_below_m"] <= c["cavitation_free_depth"]}

def propose(d, m):
    """Candidate changes (name, changes, reason) from the result m of design d against CRITERIA."""
    out, ok = [], passes(m)
    if not ok["strength"] or not ok["fatigue"]:
        out.append(("stronger root", {"chord_root_mm": round(d["chord_root_mm"] * 1.2, 1), "thickness": round(d["thickness"] + 0.02, 3)},
                    f"weakest point {m['weakest_point']}: SF yield {m['SF_yield']:.1f}, fatigue {m['SF_fatigue']:.1f} — more section at the root"))
        if d["material"] == "6061-T6":
            out.append(("NAB blades", {"material": "NAB (CuAl10Ni5Fe4)"}, "aluminium has little corrosion-fatigue strength in sea water"))
    if not ok["modes"]:
        out.append(("stiffer blade", {"thickness": round(d["thickness"] + 0.03, 3), "chord_max_mm": round(d["chord_max_mm"] * 1.1, 1)},
                    f"a wet blade mode within {m['mode_margin']:.0%} of an excitation — raise the frequency"))
    # blade count: the shaft keeps orders lcm(B, fins); pick the count whose shaft order meets the weakest wake harmonic
    hw = WAKE.harmonics(0.7, 60)
    wake_at = {b: hw[math.lcm(b, N_FINS) - 1] for b in (3, 4, 5, 6, 7)}
    best_b = min(wake_at, key=wake_at.get)
    if best_b != d["blades"] and (not ok["noise"] or not ok["hull modes"] or wake_at[best_b] < 0.5 * wake_at[d["blades"]]):
        s = d["blades"] / best_b                                          # the same blade area in more (or fewer) blades
        out.append((f"{best_b} blades", {"blades": best_b, "chord_root_mm": round(d["chord_root_mm"] * s, 1), "chord_max_mm": round(d["chord_max_mm"] * s, 1),
                                         "chord_tip_mm": round(d["chord_tip_mm"] * s, 1)},
                    f"shaft orders at {math.lcm(best_b, N_FINS)}P meet a wake harmonic of {wake_at[best_b]:.1e} instead of {wake_at[d['blades']]:.1e} "
                    f"({math.lcm(d['blades'], N_FINS)}P) — quieter, same blade area" + ("" if ok["hull modes"] else
                    f"; and the shaft order moves off the hull frequency it now meets (margin {m['hull_margin']:.0%})")))
    if not ok["cavitation"]:
        out.append(("more pitch", {"pitch_mm": round(d["pitch_mm"] * 1.12, 1)}, f"cavitates at top speed down to {m['cavitation_free_below_m']:.0f} m — the same thrust at lower rpm"))
        if d["diameter_mm"] * 1.1 <= D_MAX_MM:
            out.append(("larger diameter", {"diameter_mm": round(d["diameter_mm"] * 1.1, 1)}, "a larger, slower propeller: lower relative speed and higher efficiency"))
    if m["SF_yield"] > 3 * CRITERIA["sf_yield_min"] and m["SF_fatigue"] > 2 * CRITERIA["sf_fatigue_min"] and d["thickness"] > 0.08:
        out.append(("thinner blade", {"thickness": round(max(0.08, d["thickness"] - 0.03), 3)},
                    f"strength to spare (SF {m['SF_yield']:.0f}): thinner sections cavitate later and weigh less"))
    return out

MY_CANDIDATES = {   # your own ideas: name -> changes to DESIGN
    # "wide blades": {"chord_max_mm": 36.0},
}
base_m = evaluate(DESIGN, "eval_base")
proposals = propose(DESIGN, base_m)
pd.DataFrame([{"candidate": n, "changes": c, "reason": r} for n, c, r in proposals] + [{"candidate": n, "changes": c, "reason": "yours"} for n, c in MY_CANDIDATES.items()]).set_index("candidate")

In [ ]:
candidates = {"baseline": DESIGN} | {n: {**DESIGN, **c} for n, c, _ in proposals} | {n: {**DESIGN, **c} for n, c in MY_CANDIDATES.items()}
if len(proposals) > 1:                                                     # all the proposed changes at once
    combo = dict(DESIGN)
    for _, c, _ in proposals:
        combo.update(c)
    candidates["all proposals"] = combo
results = {"baseline": base_m}
def tag_of(name):
    return "eval_" + "".join(ch if ch.isalnum() else "_" for ch in name)
for name, d in tqdm(list(candidates.items())[1:], desc="evaluating candidates"):
    results[name] = evaluate(d, tag_of(name))
# repair rounds: a candidate that fails a criterion gets the fixes the rules propose for it (not another blade count, not an
# optimisation that would undo a repair), up to 3 times
REPAIR_ROUNDS = 3
OPTIMISATIONS = {"thinner blade"}
last = [n for n in results if n != "baseline"]
for rnd in range(1, REPAIR_ROUNDS + 1):
    new = []
    for name in [n for n in last if results[n].get("feasible") and not all(passes(results[n]).values())]:
        fixes = [c for n_, c, _ in propose(candidates[name], results[name]) if "blades" not in c and n_ not in OPTIMISATIONS]   # repairs only
        if not fixes:
            continue
        d2 = dict(candidates[name])
        for c in fixes:
            d2.update(c)
        if d2 in candidates.values():
            continue
        n2 = f"{name.split(' + fixes')[0]} + fixes" + (f" x{rnd}" if rnd > 1 else "")
        candidates[n2], results[n2] = d2, evaluate(d2, tag_of(n2))
        new.append(n2)
    if not new:
        break
    last = new
table = pd.DataFrame(results).T
ok_cols = pd.DataFrame({n: passes(m) for n, m in results.items() if m.get("feasible")}).T
table = table.join(ok_cols.add_prefix("pass_"))
table["passes_all"] = ok_cols.all(axis=1)
table["safe"] = ok_cols["strength"] & ok_cols["fatigue"]                          # hard requirements
table["criteria_passed"] = ok_cols.sum(axis=1)

def worst_ratio(m):
    """How close the design is to its weakest criterion: the smallest (value / pass mark), >= 1 when everything passes."""
    c = CRITERIA
    return min(m["SF_yield"] / c["sf_yield_min"], m["SF_fatigue"] / c["sf_fatigue_min"], m["mode_margin"] / c["mode_margin_min"],
               m["hull_margin"] / c["hull_margin_min"], c["tone_db_max"] / m["loudest_tone_db"],
               1.0 if m["cavitation_free_below_m"] <= c["cavitation_free_depth"] else c["cavitation_free_depth"] / m["cavitation_free_below_m"])

table["worst_ratio"] = pd.Series({n: worst_ratio(m) for n, m in results.items() if m.get("feasible")})
feasible = table[table["feasible"] == True].copy()
ranked = feasible.sort_values(["safe", "criteria_passed", "worst_ratio", "loudest_tone_db"], ascending=[False, False, False, True])
recommended = ranked.index[0]
show = ["blades", "diameter_mm", "pitch_mm", "thickness", "chord_root_mm", "material", "cruise_rpm", "cruise_efficiency", "endurance_h", "top_speed_m_s",
        "blade_mass_g", "SF_yield", "SF_fatigue", "first_wet_mode_hz", "mode_margin", "hull_margin", "loudest_tone_db", "total_db_cruise", "cavitation_free_below_m", "criteria_passed", "worst_ratio", "passes_all"]
failed = {n: [k for k, v in passes(results[n]).items() if not v] for n in feasible.index}
print("criteria failed by the baseline:", failed["baseline"] or "none")
if not ranked.loc[recommended, "safe"]:
    print("no candidate is strong enough — revise the criteria or add candidates to MY_CANDIDATES")
elif ranked.loc[recommended, "passes_all"]:
    print(f"recommended: {recommended} — passes every criterion" + (" (no candidate beats the baseline)" if recommended == "baseline" else ""))
else:
    print(f"recommended: {recommended} — no candidate passes every criterion; this is the closest (worst criterion at "
          f"{ranked.loc[recommended, 'worst_ratio']:.0%} of its pass mark), failing {failed[recommended]}. Add candidates to MY_CANDIDATES "
          "(for example a different pitch or diameter, which moves every excitation line) or revise CRITERIA")
ranked[show]

In [ ]:
metrics = [("cruise_electrical_W", "cruise power [W]", "lower"), ("SF_fatigue", "fatigue SF", "higher"), ("hull_margin", "hull-mode margin", "higher"),
           ("loudest_tone_db", "loudest tone at cruise [dB]", "lower"), ("cavitation_free_below_m", "cavitation-free below [m]", "lower"), ("blade_mass_g", "blade mass [g]", "lower")]
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, (col, lab, better) in zip(axes.ravel(), metrics):
    vals = feasible[col].astype(float)
    colors = ["#2e7d32" if n == recommended else ("#9e9e9e" if n == "baseline" else "#90caf9") for n in vals.index]
    ax.barh(vals.index, vals.values, color=colors); ax.set_title(f"{lab} ({better} is better)", fontsize=10); ax.grid(alpha=0.3)
fig.tight_layout()
rec = candidates[recommended]
changes = {k: v for k, v in rec.items() if DESIGN.get(k) != v}
print("to adopt the recommendation, set in section 1:\nDESIGN.update(" + ", ".join(f"{k}={v!r}" for k, v in changes.items()) + ")" if changes else "the baseline stays")

In [ ]:
# the flow of the baseline and of the recommendation side by side in one movie (slipstream model, cruise)
if recommended != "baseline":
    pr_r, sc_r, drv_r, p_r, sp_r = operating_points(rec)
    aero_r = boreas.solve(pr_r, sc_r, p_r["cruise"].rpm, (1 - W_MEAN) * sp_r["cruise"], RHO_W)
    rot_r = movie.RotorView(center=(0.0, 0.0, 0.0), diameter=pr_r.diameter, rpm=p_r["cruise"].rpm, blades=rec["blades"], inflow=(1 - W_MEAN) * sp_r["cruise"])
    s_r = wake.slipstream_sampler(pr_r, aero_r)
    s_b, r_b = model_field(pts["cruise"], (1 - W_MEAN) * speeds["cruise"])
    a = movie.make_movie(None, RUNS / "cruise_baseline.mp4", rotor=r_b, sampler=s_b, n=40, seconds=4, fps=24, title="baseline, cruise")
    b = movie.make_movie(None, RUNS / "cruise_recommended.mp4", rotor=rot_r, sampler=s_r, n=40, seconds=4, fps=24, title=f"{recommended}, cruise")
    display(Video(str(movie.concat_videos([a, b], RUNS / "baseline_vs_recommended.mp4", captions=["baseline", recommended])), embed=False, width=960))

## 11. Export

In [ ]:
doc = {"design": DESIGN, "criteria": CRITERIA, "materials": MATERIALS, "vehicle_inputs": {"L_m": L_M, "D_m": D_M, "S_wet_m2": S_WET, "wake_mean": W_MEAN, "thrust_deduction": T_DED},
       "wake": {"source": WAKE.source, "harmonics_0.7R": WAKE.harmonics(0.7, 16).tolist()},
       "operating_points": {k: {"boat_speed_m_s": speeds[k], **v.to_dict()} for k, v in pts.items()},
       "cavitation_inception_speed_m_s": inception,
       "cfd": {k: {kk: vv for kk, vv in r.metrics.items() if not isinstance(vv, (list, dict))} for k, r in CFD_OK.items()} or "not run",
       "blade_fea_top_speed": {k: v for k, v in res_blade.metrics.items() if not isinstance(v, (dict, list))}, "blade_loads": blade_loads,
       "weak_points": weak.reset_index().to_dict(orient="records"), "blade_modes_hz": {"air": f_air, "wet": f_wet},
       "hull_frequencies_hz": {"bending_estimate": f_girder, "ring": f_ring},
       "load_harmonics": {k: pd.DataFrame(h.table(30)).to_dict(orient="records") for k, h in H.items()},
       "mode_margins": margins.to_dict(orient="records"), "blade_response": response.to_dict(orient="records"), "fatigue": fatigue.to_dict(orient="records"),
       "noise": noise.to_dict(orient="index"), "candidates": {n: {"design": candidates[n], "result": results[n]} for n in candidates},
       "recommended": recommended,
       "movies": sorted(str(p) for p in RUNS.glob("*.mp4"))}
(RUNS / "submarine_propeller.json").write_text(json.dumps(doc, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("written:", RUNS / "submarine_propeller.json")

**Reading the result:** the pressure–speed plot says at what speed and depth the blades start to cavitate
(and the CFD scatter shows how much of the blade is near that line); the weak-point table says where the
blade would fail first and by how much it is safe; the frequency diagram and the margin table say whether
any fin-wake order sits on a blade or hull frequency; the noise spectra say which tone a hydrophone hears
first — behind four fins that is usually a wake tone, set by the blade count; section 10 turns all of it
into a short list of changes, each evaluated, and one recommendation you can adopt by editing `DESIGN`.